<a href="https://colab.research.google.com/github/adityab-tech/PRISMx/blob/main/notebooks/02_Fine_Tuning_Kronos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRISM - Deliverable 2--Fine-Tuning Kronos on Indian Equity Data

### Objectives

- Load Kronos Foundation Model
- Load Tokenizer
- Load Processed Dataset
- Fine-tune using LoRA
- Evaluate against Zero-Shot
- Save Best Model



#Setup

In [ ]:
!git clone https://github.com/shiyu-coder/Kronos.git

Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 371 (delta 66), reused 49 (delta 49), pack-reused 269 (from 1)
Receiving objects: 100% (371/371), 9.30 MiB | 13.61 MiB/s, done.
Resolving deltas: 100% (178/178), done.


In [ ]:
!git clone https://github.com/NeoQuasar/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [ ]:
!pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import random
import sys
import warnings
import yaml
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

warnings.filterwarnings("ignore")

In [ ]:
!pip -q install transformers peft bitsandbytes accelerate huggingface_hub sentencepiece pyyaml safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00


In [ ]:
sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/PRISM"

PROCESSED_PATH = os.path.join(PROJECT_ROOT,"data/processed")
MODEL_PATH = os.path.join(PROJECT_ROOT,"models")

#Fine Tuning

In [ ]:
from model import Kronos, KronosTokenizer
from train_sequential import SequentialTrainer
from config_loader import CustomFinetuneConfig
from finetune_base_model import create_dataloaders

In [ ]:
dataset_path = os.path.join(PROCESSED_PATH,"RELIANCE.NS.csv"
)
df = pd.read_csv(dataset_path)
df.head()

,Date,Close,High,Low,Open,Volume,Return,Log_Return,MA5,MA10,MA20,EMA20,Volatility,Volume_Change,Volume_MA20,Beta
0,2020-03-27,474.505524,493.074299,465.866824,487.597187,41657940,-0.000563,-0.000563,448.990179,443.929416,496.833708,495.441343,0.069612,-0.089334,47997996.65,1.236910
1,2020-03-30,458.853424,478.602224,454.200113,463.373180,30229866,-0.032986,-0.033542,462.028387,444.586221,490.194370,491.956779,0.069439,-0.274331,47543701.50,1.221455
2,2020-03-31,495.946472,503.093419,466.668378,478.223761,44292810,0.080839,0.077737,477.199561,449.295203,485.687991,492.336750,0.072373,0.465200,48283317.25,1.247226
3,2020-04-01,481.118164,500.777879,465.421544,499.731437,41597459,-0.029899,-0.030355,477.039258,454.280273,479.845731,491.268313,0.072290,-0.060853,48993250.30,1.235940
4,2020-04-03,479.782257,505.164047,470.364285,505.164047,41367807,-0.002777,-0.002781,478.041168,461.393848,474.006810,490.174403,0.072288,-0.005521,49956377.60,1.229398


In [ ]:
df = df.rename(
    columns={"Date":"timestamps","Open":"open","High":"high","Low":"low","Close":"close","Volume":"volume"
    })
df["amount"] = 0
df["timestamps"] = pd.to_datetime(df["timestamps"])
df.head()

,timestamps,close,high,low,open,volume,Return,Log_Return,MA5,MA10,MA20,EMA20,Volatility,Volume_Change,Volume_MA20,Beta,amount
0,2020-03-27,474.505524,493.074299,465.866824,487.597187,41657940,-0.000563,-0.000563,448.990179,443.929416,496.833708,495.441343,0.069612,-0.089334,47997996.65,1.236910,0
1,2020-03-30,458.853424,478.602224,454.200113,463.373180,30229866,-0.032986,-0.033542,462.028387,444.586221,490.194370,491.956779,0.069439,-0.274331,47543701.50,1.221455,0
2,2020-03-31,495.946472,503.093419,466.668378,478.223761,44292810,0.080839,0.077737,477.199561,449.295203,485.687991,492.336750,0.072373,0.465200,48283317.25,1.247226,0
3,2020-04-01,481.118164,500.777879,465.421544,499.731437,41597459,-0.029899,-0.030355,477.039258,454.280273,479.845731,491.268313,0.072290,-0.060853,48993250.30,1.235940,0
4,2020-04-03,479.782257,505.164047,470.364285,505.164047,41367807,-0.002777,-0.002781,478.041168,461.393848,474.006810,490.174403,0.072288,-0.005521,49956377.60,1.229398,0


In [ ]:
save_path = "/content/Kronos/finetune_csv/data/prism_train.csv"
df.to_csv(save_path,index=False)
print(save_path)

/content/Kronos/finetune_csv/data/prism_train.csv


In [ ]:
config = {

"data":{
"data_path":save_path,
"lookback_window":30,
"predict_window":5,
"max_context":30,
"clip":5,
"train_ratio":0.8,
"val_ratio":0.2,
"test_ratio":0.0
},

"training":{
"tokenizer_epochs":2,
"basemodel_epochs":2,
"batch_size":32,
"learning_rate":1e-4,
"num_workers":2
},

"model_paths":{
"exp_name":"PRISM",
"pretrained_tokenizer":"NeoQuasar/Kronos-Tokenizer-base",
"pretrained_predictor":"NeoQuasar/Kronos-base",
"base_save_path":"/content/Kronos/checkpoints"
}
}

In [ ]:
config_path = "/content/Kronos/finetune_csv/configs/prism.yaml"
with open(config_path,"w") as f:
    yaml.dump(config,f)
cfg = CustomFinetuneConfig(
    config_path)
cfg.print_config_summary()

Kronos finetuning configuration summary
Experiment name: PRISM
Data path: /content/Kronos/finetune_csv/data/prism_train.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [ ]:
train_loader, val_loader, *_ = create_dataloaders(cfg)

Creating data loaders...
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[TRAIN] Training set: first 894 time points (0.8)
[TRAIN] Training set time range: 2020-03-27 00:00:00 to 2024-02-01 00:00:00
[TRAIN] Data length after split: 894 records
[TRAIN] Data length: 894, Available samples: 859
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[VAL] Validation set: time points 895 to 1118 (0.2)
[VAL] Validation set time range: 2024-02-02 00:00:00 to 2024-12-31 00:00:00
[VAL] Data length after split: 224 records
[VAL] Data length: 224, Available samples: 189
Training set size: 859, Validation set size: 189


In [ ]:
batch_x, batch_stamp = next(iter(train_loader))

In [ ]:
print(batch_x.shape)
print(batch_stamp.shape)

torch.Size([32, 36, 6])
torch.Size([32, 36, 5])


In [ ]:
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-base")

config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
batch_x = batch_x.to(device)

In [ ]:
with torch.no_grad():
    (
        (reconstruction, encoder_output),
        bsq_loss,
        quantized,
        token_ids
    ) = tokenizer(batch_x)

In [ ]:
print(reconstruction.shape)
print(encoder_output.shape)
print(quantized.shape)
print(token_ids.shape)
print(bsq_loss)

torch.Size([32, 36, 6])
torch.Size([32, 36, 6])
torch.Size([32, 36, 20])
torch.Size([32, 36])
tensor(-0.0674)


In [ ]:
print(token_ids.min())
print(token_ids.max())
print(token_ids.unique().numel())

tensor(13741)
tensor(1024423)
338


In [ ]:
print(token_ids[:2])

tensor([[  30645,   30661,   32733,   30685,   30669,  278479,  691912,  998363,
          278415,  474079,  278415,  471979,  276399,  278415,  474031,  471979,
          459179, 1018179,  473995,  471983,  985483, 1018115,  888146, 1019219,
          675416,  998363, 1018115, 1016227,  887075,  887106,  886016,  839272,
          885828,  885792,  888146,  885074],
        [1019339,  887123,  885010, 1018251,  884994, 1018115,  998299, 1018131,
          887042,  885760,  885798, 1018243, 1018179,  887874,  885026,  884738,
         1018131,  677582,  669268,   30621,  816863,  474079,  278415,  461259,
         1018275,  278415,   30605,   30613,   30645,   16269,  278447,   30605,
           30597,  474027,  278415,   30637]])


In [ ]:
print("Tokenizer Loss :", bsq_loss.item())

Tokenizer Loss : -0.0674005001783371


In [ ]:
decoded = tokenizer.decode(token_ids)
print(decoded.shape)

torch.Size([32, 36, 6])


In [ ]:
mse = (( batch_x.cpu()-decoded.cpu())**2).mean()
print(mse.item())

0.036786999553442


In [ ]:
trainer = SequentialTrainer(config_path)
print(trainer)

Using device: cpu (rank=0, world_size=1, local_rank=0)
Kronos finetuning configuration summary
Experiment name: PRISM
Data path: /content/Kronos/finetune_csv/data/prism_train.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [ ]:
methods = [m for m in dir(trainer) if "train" in m.lower()]
print(methods)

['run_training', 'train_basemodel_phase', 'train_tokenizer_phase']


In [ ]:
print(cfg)
print(cfg.batch_size)
print(cfg.lookback_window)
print(cfg.predict_window)

32
30
5


In [ ]:
cfg.tokenizer_learning_rate = 2e-4
cfg.predictor_learning_rate = 4e-5

In [ ]:
trainer.train_tokenizer_phase()

2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - === Tokenizer Training Started ===
INFO:tokenizer_training_rank_0:=== Tokenizer Training Started ===
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Experiment Name: PRISM
INFO:tokenizer_training_rank_0:Experiment Name: PRISM
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Log Directory: /content/Kronos/checkpoints/logs
INFO:tokenizer_training_rank_0:Log Directory: /content/Kronos/checkpoints/logs
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Rank: 0
INFO:tokenizer_training_rank_0:Rank: 0
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Timestamp: 2026-07-16 09:21:18
INFO:tokenizer_training_rank_0:Timestamp: 2026-07-16 09:21:18
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Loading pretrained tokenizer...
INFO:tokenizer_training_rank_0:Loading pretrained tokenizer...



Starting Tokenizer Fine-tuning Phase
Tokenizer model exists: False
Basemodel model exists: False
Loading pretrained tokenizer...


2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Tokenizer parameters: 3,958,042
INFO:tokenizer_training_rank_0:Tokenizer parameters: 3,958,042
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - === Training Configuration ===
INFO:tokenizer_training_rank_0:=== Training Configuration ===
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Data path: /content/Kronos/finetune_csv/data/prism_train.csv
INFO:tokenizer_training_rank_0:Data path: /content/Kronos/finetune_csv/data/prism_train.csv
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Lookback window: 30
INFO:tokenizer_training_rank_0:Lookback window: 30
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Predict window: 5
INFO:tokenizer_training_rank_0:Predict window: 5
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Batch size: 32
INFO:tokenizer_training_rank_0:Batch size: 32
2026-07-16 09:21:18 - tokenizer_training_rank_0 - INFO - Learning rate: 0.0002
INFO:tokenizer_training_rank_0:Learning

Tokenizer parameters: 3,958,042
Starting tokenizer fine-tuning training...
Creating tokenizer training data loaders...
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[TRAIN] Training set: first 894 time points (0.8)
[TRAIN] Training set time range: 2020-03-27 00:00:00 to 2024-02-01 00:00:00
[TRAIN] Data length after split: 894 records
[TRAIN] Data length: 894, Available samples: 859
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[VAL] Validation set: time points 895 to 1118 (0.2)
[VAL] Validation set time range: 2024-02-02 00:00:00 to 2024-12-31 00:00:00
[VAL] Data length after split: 224 records
[VAL] Data length: 224, Available samples: 189
Training set size: 859, Validation set size: 189


2026-07-16 09:21:57 - tokenizer_training_rank_0 - INFO - 
--- Epoch 1/2 Summary ---
Validation Loss: 0.0252
Epoch Time: 0:00:32
Total Training Time: 0:00:32

INFO:tokenizer_training_rank_0:
--- Epoch 1/2 Summary ---
Validation Loss: 0.0252
Epoch Time: 0:00:32
Total Training Time: 0:00:32

2026-07-16 09:21:57 - tokenizer_training_rank_0 - INFO - Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0252)
INFO:tokenizer_training_rank_0:Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0252)


...

2026-07-16 09:22:25 - tokenizer_training_rank_0 - INFO - [Epoch 2/2, Step 24/26] LR: 0.000000, Loss: -0.0079
INFO:tokenizer_training_rank_0:[Epoch 2/2, Step 24/26] LR: 0.000000, Loss: -0.0079
2026-07-16 09:22:25 - tokenizer_training_rank_0 - INFO -   - VQ Loss: -0.0674
  - Recon Loss Pre: 0.0324
  - Recon Loss All: 0.0193
INFO:tokenizer_training_rank_0:  - VQ Loss: -0.0674
  - Recon Loss Pre: 0.0324
  - Recon Loss All: 0.0193


[Epoch 2/2, Step 24/26] LR: 0.000000, Loss: -0.0079
  - VQ Loss: -0.0674
  - Recon Loss Pre: 0.0324
  - Recon Loss All: 0.0193


2026-07-16 09:22:29 - tokenizer_training_rank_0 - INFO - 
--- Epoch 2/2 Summary ---
Validation Loss: 0.0242
Epoch Time: 0:00:31
Total Training Time: 0:00:31

INFO:tokenizer_training_rank_0:
--- Epoch 2/2 Summary ---
Validation Loss: 0.0242
Epoch Time: 0:00:31
Total Training Time: 0:00:31

2026-07-16 09:22:29 - tokenizer_training_rank_0 - INFO - Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0242)
INFO:tokenizer_training_rank_0:Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0242)
2026-07-16 09:22:29 - tokenizer_training_rank_0 - INFO - Tokenizer training completed! Best validation loss: 0.0242
Training time: 1.19 minutes
Model saved to: /content/Kronos/checkpoints/tokenizer
INFO:tokenizer_training_rank_0:Tokenizer training completed! Best validation loss: 0.0242
Training time: 1.19 minutes
Model saved to: /content/Kronos/checkpoints/tokenizer



--- Epoch 2/2 Summary ---
Validation Loss: 0.0242
Epoch Time: 0:00:31
Total Training Time: 0:00:31

Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0242)

Tokenizer training completed! Best validation loss: 0.0242
Training time: 1.19 minutes
Model saved to: /content/Kronos/checkpoints/tokenizer


True

In [ ]:
print("Tokenizer save path :", cfg.tokenizer_save_path)
print("Basemodel save path :", cfg.basemodel_save_path)
print("Base save path :", cfg.base_save_path)

Tokenizer save path : /content/Kronos/checkpoints/tokenizer
Basemodel save path : /content/Kronos/checkpoints/basemodel
Base save path : /content/Kronos/checkpoints


In [ ]:
print("Tokenizer dir exists:", os.path.exists(cfg.tokenizer_save_path))
print("Basemodel dir exists:", os.path.exists(cfg.basemodel_save_path))

if os.path.exists(cfg.tokenizer_save_path):
    print(os.listdir(cfg.tokenizer_save_path))

Tokenizer dir exists: True
Basemodel dir exists: False
['best_model']


In [ ]:
config["model_paths"]["finetuned_tokenizer"] = "/content/Kronos/checkpoints/tokenizer/best_model"
config["model_paths"]["tokenizer_save_name"] = "tokenizer"
config["model_paths"]["basemodel_save_name"] = "basemodel"

In [ ]:
with open(config_path, "w") as f:
    yaml.dump(config, f)
cfg=CustomFinetuneConfig(config_path)

In [ ]:
print("finetuned_tokenizer_path :", cfg.finetuned_tokenizer_path)
print("tokenizer_save_path      :", cfg.tokenizer_save_path)
print("basemodel_save_path      :", cfg.basemodel_save_path)

finetuned_tokenizer_path : /content/Kronos/checkpoints/tokenizer/best_model
tokenizer_save_path      : /content/Kronos/checkpoints/tokenizer
basemodel_save_path      : /content/Kronos/checkpoints/basemodel


In [ ]:
trainer = SequentialTrainer(config_path)

Using device: cpu (rank=0, world_size=1, local_rank=0)
Kronos finetuning configuration summary
Experiment name: PRISM
Data path: /content/Kronos/finetune_csv/data/prism_train.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [ ]:
trainer.train_basemodel_phase()

2026-07-16 09:32:42 - basemodel_training_rank_0 - INFO - === Basemodel Training Started ===
INFO:basemodel_training_rank_0:=== Basemodel Training Started ===
2026-07-16 09:32:42 - basemodel_training_rank_0 - INFO - Experiment Name: PRISM
INFO:basemodel_training_rank_0:Experiment Name: PRISM
2026-07-16 09:32:42 - basemodel_training_rank_0 - INFO - Log Directory: /content/Kronos/checkpoints/logs
INFO:basemodel_training_rank_0:Log Directory: /content/Kronos/checkpoints/logs
2026-07-16 09:32:42 - basemodel_training_rank_0 - INFO - Rank: 0
INFO:basemodel_training_rank_0:Rank: 0
2026-07-16 09:32:42 - basemodel_training_rank_0 - INFO - Timestamp: 2026-07-16 09:32:42
INFO:basemodel_training_rank_0:Timestamp: 2026-07-16 09:32:42
2026-07-16 09:32:42 - basemodel_training_rank_0 - INFO - Loading fine-tuned tokenizer...
INFO:basemodel_training_rank_0:Loading fine-tuned tokenizer...
2026-07-16 09:32:42 - basemodel_training_rank_0 - INFO - Loading pretrained predictor...
INFO:basemodel_training_rank_


Starting Basemodel Fine-tuning Phase
Tokenizer model exists: True
Basemodel model exists: False
Loading fine-tuned tokenizer...
Loading weights from local directory
Loading pretrained predictor...


2026-07-16 09:32:44 - basemodel_training_rank_0 - INFO - Model parameters: 102,310,592
INFO:basemodel_training_rank_0:Model parameters: 102,310,592
2026-07-16 09:32:44 - basemodel_training_rank_0 - INFO - === Training Configuration ===
INFO:basemodel_training_rank_0:=== Training Configuration ===
2026-07-16 09:32:44 - basemodel_training_rank_0 - INFO - Data path: /content/Kronos/finetune_csv/data/prism_train.csv
INFO:basemodel_training_rank_0:Data path: /content/Kronos/finetune_csv/data/prism_train.csv
2026-07-16 09:32:44 - basemodel_training_rank_0 - INFO - Lookback window: 30
INFO:basemodel_training_rank_0:Lookback window: 30
2026-07-16 09:32:44 - basemodel_training_rank_0 - INFO - Predict window: 5
INFO:basemodel_training_rank_0:Predict window: 5
2026-07-16 09:32:44 - basemodel_training_rank_0 - INFO - Batch size: 32
INFO:basemodel_training_rank_0:Batch size: 32
2026-07-16 09:32:44 - basemodel_training_rank_0 - INFO - Learning rate: 4e-05
INFO:basemodel_training_rank_0:Learning rate

Model parameters: 102,310,592
Starting fine-tuning training...
Creating data loaders...
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[TRAIN] Training set: first 894 time points (0.8)
[TRAIN] Training set time range: 2020-03-27 00:00:00 to 2024-02-01 00:00:00
[TRAIN] Data length after split: 894 records
[TRAIN] Data length: 894, Available samples: 859
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[VAL] Validation set: time points 895 to 1118 (0.2)
[VAL] Validation set time range: 2024-02-02 00:00:00 to 2024-12-31 00:00:00
[VAL] Data length after split: 224 records
[VAL] Data length: 224, Available samples: 189
Training set size: 859, Validation set size: 189


2026-07-16 09:38:50 - basemodel_training_rank_0 - INFO - 
--- Epoch 1/2 Summary ---
Training Loss: 3.4466
Validation Loss: 3.7212
Epoch Time: 365.94 seconds

INFO:basemodel_training_rank_0:
--- Epoch 1/2 Summary ---
Training Loss: 3.4466
Validation Loss: 3.7212
Epoch Time: 365.94 seconds



...

2026-07-16 09:45:05 - basemodel_training_rank_0 - INFO - 
--- Epoch 2/2 Summary ---
Training Loss: 3.2586
Validation Loss: 3.7007
Epoch Time: 369.05 seconds

INFO:basemodel_training_rank_0:
--- Epoch 2/2 Summary ---
Training Loss: 3.2586
Validation Loss: 3.7007
Epoch Time: 369.05 seconds




--- Epoch 2/2 Summary ---
Training Loss: 3.2586
Validation Loss: 3.7007
Epoch Time: 369.05 seconds



2026-07-16 09:45:09 - basemodel_training_rank_0 - INFO - Best model saved to: /content/Kronos/checkpoints/basemodel/best_model (validation loss: 3.7007)
INFO:basemodel_training_rank_0:Best model saved to: /content/Kronos/checkpoints/basemodel/best_model (validation loss: 3.7007)
2026-07-16 09:45:10 - basemodel_training_rank_0 - INFO - Basemodel training completed! Best validation loss: 3.7007
Training time: 12.43 minutes
Model saved to: /content/Kronos/checkpoints/basemodel
INFO:basemodel_training_rank_0:Basemodel training completed! Best validation loss: 3.7007
Training time: 12.43 minutes
Model saved to: /content/Kronos/checkpoints/basemodel


Best model saved to: /content/Kronos/checkpoints/basemodel/best_model (validation loss: 3.7007)

Basemodel training completed! Best validation loss: 3.7007
Training time: 12.43 minutes
Model saved to: /content/Kronos/checkpoints/basemodel


True

In [ ]:
import inspect

source = inspect.getsource(SequentialTrainer.train_basemodel_phase)
print(source)

    def train_basemodel_phase(self):
        print("\n" + "="*60)
        print("Starting Basemodel Fine-tuning Phase")
        print("="*60)
        
        if getattr(self.config, 'pre_trained_tokenizer', True):
            if not os.path.exists(self.config.finetuned_tokenizer_path):
                raise FileNotFoundError(f"Fine-tuned tokenizer does not exist: {self.config.finetuned_tokenizer_path}")
        
        _, basemodel_exists = self._check_existing_models()
        if basemodel_exists and self.config.skip_existing:
            print("Basemodel model already exists, skipping training")
            return True
        
        log_dir = os.path.join(self.config.base_save_path, "logs")
        logger = setup_basemodel_logging(self.config.exp_name, log_dir, self.rank)
        
        set_seed(self.config.seed)
        
        if getattr(self.config, 'pre_trained_tokenizer', True):
            logger.info("Loading fine-tuned tokenizer...")
            if self.rank == 0:
   

In [ ]:
print("Tokenizer:")
print(os.listdir("/content/Kronos/checkpoints/tokenizer"))
print("Basemodel:")
print(os.listdir("/content/Kronos/checkpoints/basemodel"))

Tokenizer:
['best_model']
Basemodel:
['best_model']


In [ ]:
print(os.path.exists("/content/Kronos/checkpoints/tokenizer/best_model"))
print(os.path.exists("/content/Kronos/checkpoints/basemodel/best_model"))

True
True


In [ ]:
drive_model_dir = os.path.join( PROJECT_ROOT, "models", "kronos")
os.makedirs(drive_model_dir, exist_ok=True)

shutil.copytree(
    "/content/Kronos/checkpoints/tokenizer",os.path.join(drive_model_dir, "tokenizer"),dirs_exist_ok=True)

shutil.copytree(
    "/content/Kronos/checkpoints/basemodel",os.path.join(drive_model_dir, "basemodel"),dirs_exist_ok=True)

print("Models copied successfully.")

Models copied successfully.


In [ ]:
print("PRISM Deliverable 2 Completed Successfully")

PRISM Deliverable 2 Completed Successfully


#LoRA/QLoRA

In [129]:
tokenizer_ckpt = "/content/Kronos/checkpoints/tokenizer/best_model"
basemodel_ckpt = "/content/Kronos/checkpoints/basemodel/best_model"

print("Tokenizer Exists :", os.path.exists(tokenizer_ckpt))
print("Basemodel Exists :", os.path.exists(basemodel_ckpt))

Tokenizer Exists : True
Basemodel Exists : True


In [130]:
from model.kronos import Kronos, KronosTokenizer

finetuned_tokenizer = KronosTokenizer.from_pretrained(tokenizer_ckpt)
finetuned_model = Kronos.from_pretrained(basemodel_ckpt)

finetuned_model = finetuned_model.to(device)
finetuned_model.eval()

print("Fine-tuned models loaded successfully.")

Loading weights from local directory
Loading weights from local directory
Fine-tuned models loaded successfully.


In [131]:
print(type(finetuned_tokenizer))
print(type(finetuned_model))

<class 'model.kronos.KronosTokenizer'>
<class 'model.kronos.Kronos'>


In [132]:
batch_x, batch_stamp = next(iter(train_loader))

batch_x = batch_x.to(device)
batch_stamp = batch_stamp.to(device)

with torch.no_grad():
    token_seq_0, token_seq_1 = finetuned_tokenizer.encode(
        batch_x,
        half=True
    )

print(token_seq_0.shape)
print(token_seq_1.shape)

torch.Size([32, 36])
torch.Size([32, 36])


In [133]:
with torch.no_grad():

    s1_logits, s2_logits = finetuned_model(
        token_seq_0,
        token_seq_1,
        batch_stamp
    )

print("S1 logits :", s1_logits.shape)
print("S2 logits :", s2_logits.shape)

S1 logits : torch.Size([32, 36, 1024])
S2 logits : torch.Size([32, 36, 1024])


In [134]:
total_params = sum(p.numel() for p in finetuned_model.parameters())
trainable_params = sum(
    p.numel() for p in finetuned_model.parameters()
    if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 102,310,592
Trainable Parameters : 102,310,592


In [135]:
models_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)

shutil.copytree(
    tokenizer_ckpt,
    os.path.join(models_dir, "tokenizer"),
    dirs_exist_ok=True
)

shutil.copytree(
    basemodel_ckpt,
    os.path.join(models_dir, "basemodel"),
    dirs_exist_ok=True
)

print("Models saved successfully.")

Models saved successfully.


In [137]:
print(hasattr(finetuned_tokenizer, "decode"))
print(inspect.signature(finetuned_tokenizer.decode))
print(inspect.getsource(finetuned_tokenizer.decode))

True
(x, half=False)
    def decode(self, x, half=False):
        """
        Decodes quantized indices back to the input data space.

        Args:
            x (torch.Tensor): Quantized indices tensor.
            half (bool, optional): Whether the indices were generated with half quantization. Defaults to False.

        Returns:
            torch.Tensor: Reconstructed output tensor of shape (batch_size, seq_len, d_in).
        """
        quantized = self.indices_to_bits(x, half)
        z = self.post_quant_embed(quantized)
        for layer in self.decoder:
            z = layer(z)
        z = self.head(z)
        return z



In [138]:
zero_tokenizer = KronosTokenizer.from_pretrained(
    "NeoQuasar/Kronos-Tokenizer-base")

zero_model = Kronos.from_pretrained(
    "NeoQuasar/Kronos-base").to(device)

zero_model.eval()

Kronos(
  (token_drop): Dropout(p=0.0, inplace=False)
  (embedding): HierarchicalEmbedding(
    (emb_s1): Embedding(1024, 832)
    (emb_s2): Embedding(1024, 832)
    (fusion_proj): Linear(in_features=1664, out_features=832, bias=True)
  )
  (time_emb): TemporalEmbedding(
    (minute_embed): Embedding(60, 832)
    (hour_embed): Embedding(24, 832)
    (weekday_embed): Embedding(7, 832)
    (day_embed): Embedding(32, 832)
    (month_embed): Embedding(13, 832)
  )
  (transformer): ModuleList(
    (0-11): 12 x TransformerBlock(
      (norm1): RMSNorm()
      (self_attn): MultiHeadAttentionWithRoPE(
        (q_proj): Linear(in_features=832, out_features=832, bias=True)
        (k_proj): Linear(in_features=832, out_features=832, bias=True)
        (v_proj): Linear(in_features=832, out_features=832, bias=True)
        (out_proj): Linear(in_features=832, out_features=832, bias=True)
        (rotary): RotaryPositionalEmbedding()
        (resid_dropout): Dropout(p=0.2, inplace=False)
      )
    

In [139]:
batch_x, batch_stamp = next(iter(val_loader))

batch_x = batch_x.to(device)
batch_stamp = batch_stamp.to(device)

In [141]:
for name, obj in inspect.getmembers(Kronos, inspect.isfunction):
    print(name)

__call__
__delattr__
__dir__
__getattr__
__getstate__
__init__
__new__
__repr__
__setattr__
__setstate__
_apply
_call_impl
_get_backward_hooks
_get_backward_pre_hooks
_get_name
_init_weights
_load_from_state_dict
_maybe_warn_non_full_backward_hook
_named_members
_register_load_state_dict_pre_hook
_register_state_dict_hook
_replicate_for_data_parallel
_save_pretrained
_save_to_state_dict
_slow_forward
_wrapped_call_impl
add_module
apply
bfloat16
buffers
children
compile
cpu
cuda
decode_s1
decode_s2
double
eval
extra_repr
float
forward
generate_model_card
get_buffer
get_extra_state
get_parameter
get_submodule
half
ipu
load_state_dict
modules
mtia
named_buffers
named_children
named_modules
named_parameters
parameters
push_to_hub
register_backward_hook
register_buffer
register_forward_hook
register_forward_pre_hook
register_full_backward_hook
register_full_backward_pre_hook
register_load_state_dict_post_hook
register_load_state_dict_pre_hook
register_module
register_parameter
register_stat

In [140]:
with torch.no_grad():
    s1_ids, s2_ids = zero_tokenizer.encode(
        batch_x,
        half=True
    )

    s1_logits, s2_logits = zero_model(
        s1_ids,
        s2_ids,
        batch_stamp
    )

    pred_tokens = torch.argmax(s2_logits, dim=-1)

    zero_prediction = zero_tokenizer.decode(
        pred_tokens,
        half=True
    )

print(zero_prediction.shape)

ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
with torch.no_grad():

    s1_ids, s2_ids = finetuned_tokenizer.encode(
        batch_x,
        half=True
    )

    s1_logits, s2_logits = finetuned_model(
        s1_ids,
        s2_ids,
        batch_stamp
    )

    pred_tokens = torch.argmax(s2_logits, dim=-1)

    finetuned_prediction = finetuned_tokenizer.decode(
        pred_tokens,
        half=True
    )

print(finetuned_prediction.shape)